In [5]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from pathlib import Path

# -----------------------
# constants
# -----------------------
mu_sun = 1.32712440018e11  # km^3/s^2 (your value)
mu_earth = 398600.4418     # km^3/s^2
r_earth = 6378.1363        # km
au_km = 149597870.7        # km

# -----------------------
# mission setup
# -----------------------
alt_leo = 200.0  # km
r_leo = r_earth + alt_leo

r1 = 1.0 * au_km
r2 = 5.2044 * au_km  # Jupiter mean semi-major axis (AU)

# -----------------------
# heliocentric hohmann + patched conic departure metrics
# -----------------------
v1 = np.sqrt(mu_sun / r1)
v2 = np.sqrt(mu_sun / r2)

a_t = 0.5 * (r1 + r2)
e_t = (r2 - r1) / (r2 + r1)
p_t = a_t * (1 - e_t**2)

v_t1 = np.sqrt(mu_sun * (2.0 / r1 - 1.0 / a_t))
v_t2 = np.sqrt(mu_sun * (2.0 / r2 - 1.0 / a_t))

dv_helio_1 = v_t1 - v1
dv_helio_2 = v2 - v_t2
dv_helio_total = dv_helio_1 + dv_helio_2

tof_sec = np.pi * np.sqrt(a_t**3 / mu_sun)
tof_days = tof_sec / 86400.0
tof_years = tof_days / 365.25

v_inf_earth = dv_helio_1
v_inf_jupiter = abs(dv_helio_2)

v_circ_leo = np.sqrt(mu_earth / r_leo)
v_esc_leo = np.sqrt(2.0 * mu_earth / r_leo)
v_perigee_hyp = np.sqrt(v_inf_earth**2 + v_esc_leo**2)
dv_leo_depart = v_perigee_hyp - v_circ_leo
C3 = v_inf_earth**2

# -----------------------
# print report
# -----------------------
print("earth leo -> jupiter (heliocentric) hohmann (coplanar, circular, impulsive)")
print("------------------------------------------------------------")

print("inputs / constants:")
print(f"  alt leo:                 {alt_leo:,.1f} km")
print(f"  r_earth:                 {r_earth:,.4f} km")
print(f"  mu_earth:                {mu_earth:,.4f} km^3/s^2")
print(f"  mu_sun:                  {mu_sun:,.4f} km^3/s^2")
print(f"  1 au:                    {au_km:,.1f} km")
print()

print("basic geometry:")
print(f"  r_leo:                   {r_leo:,.1f} km")
print(f"  r1 (earth orbit):        {r1/au_km:.6f} au   ({r1:,.0f} km)")
print(f"  r2 (jupiter orbit):      {r2/au_km:.6f} au   ({r2:,.0f} km)")
print()

print("heliocentric circular speeds:")
print(f"  v_earth (1 au):          {v1:,.3f} km/s")
print(f"  v_jupiter (~5.2 au):     {v2:,.3f} km/s")
print()

print("transfer ellipse (sun-centered):")
print(f"  a_t:                     {a_t/au_km:.6f} au   ({a_t:,.0f} km)")
print(f"  e_t:                     {e_t:.6f}")
print(f"  p_t:                     {p_t/au_km:.6f} au   ({p_t:,.0f} km)")
print(f"  v_t at r1 (perihelion):  {v_t1:,.3f} km/s")
print(f"  v_t at r2 (aphelion):    {v_t2:,.3f} km/s")
print()

print("impulsive hohmann deltas (heliocentric):")
print(f"  dv1 at earth:            {dv_helio_1:,.3f} km/s")
print(f"  dv2 at jupiter:          {dv_helio_2:,.3f} km/s")
print(f"  dv total (helio):        {dv_helio_total:,.3f} km/s")
print()

print("time of flight:")
print(f"  tof (half-ellipse):      {tof_days:,.1f} days  ({tof_years:.3f} yr)")
print()

print("patched-conic earth departure from 200 km leo:")
print(f"  v_inf,earth:             {v_inf_earth:,.3f} km/s")
print(f"  C3:                      {C3:,.2f} km^2/s^2")
print(f"  v_circ leo:              {v_circ_leo:,.3f} km/s")
print(f"  v_escape leo:            {v_esc_leo:,.3f} km/s")
print(f"  v_perigee hyperbola:     {v_perigee_hyp:,.3f} km/s")
print(f"  dv from leo:             {dv_leo_depart:,.3f} km/s")
print()

print("jupiter arrival (no capture modeled):")
print(f"  v_inf,jupiter:           {v_inf_jupiter:,.3f} km/s")

# -----------------------
# animation setup (same depart/arrive points as your static plot)
# -----------------------
n_earth = np.sqrt(mu_sun / r1**3)
n_jup = np.sqrt(mu_sun / r2**3)

# phases so: earth departs at +x (theta=0) and jupiter is at -x (theta=pi) at t=tof
theta_e0 = 0.0
theta_j0 = np.pi - n_jup * tof_sec

# transfer path (true anomaly from 0 to pi)
nu = np.linspace(0.0, np.pi, 900)
r_nu = p_t / (1 + e_t * np.cos(nu))
x_tr = r_nu * np.cos(nu)
y_tr = r_nu * np.sin(nu)

# orbit circles
th = np.linspace(0.0, 2.0 * np.pi, 2400)
x_e_orb = r1 * np.cos(th)
y_e_orb = r1 * np.sin(th)
x_j_orb = r2 * np.cos(th)
y_j_orb = r2 * np.sin(th)

# animation timeline
N = 220
t = np.linspace(0.0, tof_sec, N)

theta_e = theta_e0 + n_earth * t
theta_j = theta_j0 + n_jup * t

x_e = r1 * np.cos(theta_e)
y_e = r1 * np.sin(theta_e)

x_j = r2 * np.cos(theta_j)
y_j = r2 * np.sin(theta_j)

# spacecraft motion: walk along transfer ellipse in sync with time (visual only)
idx_nu = np.linspace(0, len(nu) - 1, N).astype(int)
x_sc = x_tr[idx_nu]
y_sc = y_tr[idx_nu]

# convert to AU for plotting
to_au = 1.0 / au_km
x_tr_au, y_tr_au = x_tr * to_au, y_tr * to_au
x_e_orb_au, y_e_orb_au = x_e_orb * to_au, y_e_orb * to_au
x_j_orb_au, y_j_orb_au = x_j_orb * to_au, y_j_orb * to_au

x_e_au, y_e_au = x_e * to_au, y_e * to_au
x_j_au, y_j_au = x_j * to_au, y_j * to_au
x_sc_au, y_sc_au = x_sc * to_au, y_sc * to_au


def wrap_pi(a):
    return (a + np.pi) % (2.0 * np.pi) - np.pi


# -----------------------
# build gif
# -----------------------
fig, ax = plt.subplots()
ax.set_aspect("equal", adjustable="box")
ax.set_xlabel("x (au)")
ax.set_ylabel("y (au)")
ax.set_title("earth -> jupiter hohmann")
ax.grid(True, alpha=0.3)

# static context
ax.plot(0, 0, marker="o", markersize=6)
ax.text(0, 0, "  sun", va="center")

ax.plot(x_e_orb_au, y_e_orb_au, label="earth orbit (1 au)")
ax.plot(x_j_orb_au, y_j_orb_au, label="jupiter orbit (~5.2 au)")
ax.plot(x_tr_au, y_tr_au, label="transfer ellipse")

# fixed depart/arrive points
ax.plot([r1 * to_au], [0], marker="x", markersize=7)
ax.text(r1 * to_au, 0, "  depart", va="bottom")

ax.plot([-r2 * to_au], [0], marker="x", markersize=7)
ax.text(-r2 * to_au, 0, "  arrive", va="bottom", ha="right")

# animated artists
earth_dot, = ax.plot([], [], marker="o", markersize=5)
jup_dot, = ax.plot([], [], marker="o", markersize=6)
sc_dot, = ax.plot([], [], marker="o", markersize=4)
sc_trail, = ax.plot([], [], linewidth=1.5)

phase_text = ax.text(0.02, 0.02, "", transform=ax.transAxes)

pad = 0.7
ax.set_xlim(-(r2 * to_au + pad), (r2 * to_au + pad))
ax.set_ylim(-(r2 * to_au + pad), (r2 * to_au + pad))
ax.legend(loc="upper right")


def update(k):
    earth_dot.set_data([x_e_au[k]], [y_e_au[k]])
    jup_dot.set_data([x_j_au[k]], [y_j_au[k]])
    sc_dot.set_data([x_sc_au[k]], [y_sc_au[k]])
    sc_trail.set_data(x_sc_au[: k + 1], y_sc_au[: k + 1])

    dtheta = wrap_pi(theta_j[k] - theta_e[k])
    phase_deg = np.degrees(dtheta)
    days = t[k] / 86400.0
    phase_text.set_text(f"t = {days:6.0f} days\nphase (jup - earth) = {phase_deg:6.1f} deg")

    return earth_dot, jup_dot, sc_dot, sc_trail, phase_text


anim = FuncAnimation(fig, update, frames=N, interval=50, blit=True)

out_path = Path("earth_leo_to_jupiter_hohmann.gif")
anim.save(out_path, writer=PillowWriter(fps=20))

plt.close(fig)
print(f"wrote: {out_path.resolve()}")

earth leo -> jupiter (heliocentric) hohmann (coplanar, circular, impulsive)
------------------------------------------------------------
inputs / constants:
  alt leo:                 200.0 km
  r_earth:                 6,378.1363 km
  mu_earth:                398,600.4418 km^3/s^2
  mu_sun:                  132,712,440,018.0000 km^3/s^2
  1 au:                    149,597,870.7 km

basic geometry:
  r_leo:                   6,578.1 km
  r1 (earth orbit):        1.000000 au   (149,597,871 km)
  r2 (jupiter orbit):      5.204400 au   (778,567,158 km)

heliocentric circular speeds:
  v_earth (1 au):          29.785 km/s
  v_jupiter (~5.2 au):     13.056 km/s

transfer ellipse (sun-centered):
  a_t:                     3.102200 au   (464,082,514 km)
  e_t:                     0.677648
  p_t:                     1.677648 au   (250,972,587 km)
  v_t at r1 (perihelion):  38.578 km/s
  v_t at r2 (aphelion):    7.413 km/s

impulsive hohmann deltas (heliocentric):
  dv1 at earth:            8.79

In [ ]:
# CELL 4: EEJ ideal patched conic — analysis + animation

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from pathlib import Path

# -----------------------
# constants
# -----------------------
mu_sun   = 1.32712440018e11   # km^3/s^2
mu_earth = 398600.4418
r_earth  = 6378.1363
au_km    = 149597870.7

alt_leo          = 200.0
r_leo            = r_earth + alt_leo
r_flyby_alt_EEJ  = 300.0              # km above Earth surface for flyby
r_flyby_EEJ      = r_earth + r_flyby_alt_EEJ

r_E = 1.0000 * au_km
r_J = 5.2044 * au_km

v_E = np.sqrt(mu_sun / r_E)
v_J = np.sqrt(mu_sun / r_J)

v_circ_leo = np.sqrt(mu_earth / r_leo)
v_esc_leo  = np.sqrt(2.0 * mu_earth / r_leo)

# -----------------------
# EEJ PHYSICS
#
# Leg 2 is identical to direct Hohmann (Earth r → Jupiter r).
# The flyby substitutes for the departure energy: instead of burning
# to v_leg2_peri directly, we burn less, loop back to Earth, and
# let the gravity assist make up the difference.
#
# Max-energy flyby geometry: v_inf rotated to align prograde with Earth
#   v_sc_post = v_E + v_inf → v_inf = v_leg2_peri - v_E
# Resonant Leg 1: inner ellipse (sc slower than Earth), same |v_inf|
#   sc dips inward and returns to Earth after one full period
# -----------------------

# Leg 2 (Earth → Jupiter, same as direct Hohmann)
a_leg2   = 0.5 * (r_E + r_J)
e_leg2   = (r_J - r_E) / (r_J + r_E)
v_leg2_peri = np.sqrt(mu_sun * (2.0/r_E - 1.0/a_leg2))
tof_leg2_sec  = np.pi * np.sqrt(a_leg2**3 / mu_sun)
tof_leg2_days = tof_leg2_sec / 86400.0

# v_inf at flyby
v_inf_flyby = v_leg2_peri - v_E

# Flyby bend angle
sin_half_d  = mu_earth / (r_flyby_EEJ * v_inf_flyby**2 + mu_earth)
delta_deg   = np.degrees(2.0 * np.arcsin(np.clip(sin_half_d, -1, 1)))

# Leg 1: inner resonant ellipse
# sc departs slower than Earth → aphelion at r_E, dips inward, returns
v_sc_depart  = v_E - v_inf_flyby
a_leg1       = 1.0 / (2.0/r_E - v_sc_depart**2 / mu_sun)
r_peri_leg1  = 2.0 * a_leg1 - r_E
e_leg1       = (r_E - r_peri_leg1) / (r_E + r_peri_leg1)
T_leg1_sec   = 2.0 * np.pi * np.sqrt(a_leg1**3 / mu_sun)
tof_leg1_days = T_leg1_sec / 86400.0

# Total
tof_total_days  = tof_leg1_days + tof_leg2_days
tof_total_years = tof_total_days / 365.25

# Departure burn from LEO
C3_EEJ        = v_inf_flyby**2
v_perigee_EEJ = np.sqrt(C3_EEJ + v_esc_leo**2)
dv_EEJ        = v_perigee_EEJ - v_circ_leo

# Jupiter arrival
v_arr_J  = np.sqrt(mu_sun * (2.0/r_J - 1.0/a_leg2))
v_inf_J  = abs(v_arr_J - v_J)

# Reference: direct Hohmann
v_inf_direct     = v_leg2_peri - v_E   # same as v_inf_flyby — note below
v_perigee_direct = np.sqrt(v_inf_direct**2 + v_esc_leo**2)
dv_direct        = v_perigee_direct - v_circ_leo
tof_direct_days  = tof_leg2_days  # direct = just Leg 2

# -----------------------
# PRINT REPORT
# -----------------------
print("earth leo -> jupiter via earth flyby (EEJ) — ideal patched conic")
print("------------------------------------------------------------")
print()
print("inputs / constants:")
print(f"  alt leo:                 {alt_leo:,.1f} km")
print(f"  earth flyby altitude:    {r_flyby_alt_EEJ:,.1f} km  (r = {r_flyby_EEJ:,.1f} km)")
print(f"  r1 (earth orbit):        {r_E/au_km:.4f} au")
print(f"  r2 (jupiter orbit):      {r_J/au_km:.4f} au")
print()
print("leg 1 — inner resonant orbit (earth → earth flyby):")
print(f"  departure speed:         {v_sc_depart:.3f} km/s heliocentric (slower than earth)")
print(f"  departure v_inf:         {v_inf_flyby:.3f} km/s")
print(f"  C3:                      {C3_EEJ:.2f} km²/s²")
print(f"  semi-major axis:         {a_leg1/au_km:.4f} au")
print(f"  eccentricity:            {e_leg1:.4f}")
print(f"  perihelion:              {r_peri_leg1/au_km:.4f} au  ({r_peri_leg1:,.0f} km)")
print(f"  orbital period (leg 1):  {tof_leg1_days:.1f} days  ({tof_leg1_days/365.25:.3f} yr)")
print()
print(f"earth gravity assist (flyby at {r_flyby_alt_EEJ:.0f} km altitude):")
print(f"  flyby v_inf:             {v_inf_flyby:.3f} km/s")
print(f"  bend angle delta:        {delta_deg:.1f} deg")
print(f"  helio speed before:      {v_sc_depart:.3f} km/s")
print(f"  helio speed after:       {v_leg2_peri:.3f} km/s")
print(f"  delta-v equivalent:      {v_leg2_peri - v_sc_depart:.3f} km/s  (free from flyby)")
print()
print("leg 2 — earth flyby → jupiter (same as hohmann):")
print(f"  semi-major axis:         {a_leg2/au_km:.4f} au")
print(f"  eccentricity:            {e_leg2:.4f}")
print(f"  tof (leg 2):             {tof_leg2_days:.1f} days  ({tof_leg2_days/365.25:.3f} yr)")
print(f"  v_inf at jupiter:        {v_inf_J:.3f} km/s")
print()
print("patched-conic earth departure from 200 km leo:")
print(f"  v_inf earth:             {v_inf_flyby:.3f} km/s")
print(f"  C3:                      {C3_EEJ:.2f} km²/s²")
print(f"  v_circ leo:              {v_circ_leo:.3f} km/s")
print(f"  v_escape leo:            {v_esc_leo:.3f} km/s")
print(f"  v_perigee hyperbola:     {v_perigee_EEJ:.3f} km/s")
print(f"  dv from leo:             {dv_EEJ:.3f} km/s")
print()
print("totals:")
print(f"  total tof:               {tof_total_days:.1f} days  ({tof_total_years:.2f} yr)")
print(f"  dv from leo:             {dv_EEJ:.3f} km/s")
print()
print("comparison — direct hohmann:")
print(f"  dv from leo:             {dv_direct:.3f} km/s")
print(f"  tof:                     {tof_direct_days:.1f} days  ({tof_direct_days/365.25:.2f} yr)")
print(f"  dv savings (EEJ):        {dv_direct - dv_EEJ:+.3f} km/s")
print(f"  tof penalty (EEJ):       {tof_total_days - tof_direct_days:+.1f} days  "
      f"({(tof_total_days - tof_direct_days)/365.25:+.2f} yr)")

# -----------------------
# ANIMATION
# -----------------------
n_E = np.sqrt(mu_sun / r_E**3)
n_J = np.sqrt(mu_sun / r_J**3)
to_au = 1.0 / au_km

p_leg1 = a_leg1 * (1 - e_leg1**2)
p_leg2 = a_leg2 * (1 - e_leg2**2)

# Leg 1: full inner loop — aphelion at +x (departure), goes inward and back
nu1 = np.linspace(np.pi, 3*np.pi, 900)   # aphelion → perihelion → aphelion
r_nu1 = p_leg1 / (1 + e_leg1 * np.cos(nu1))
x_leg1 = r_nu1 * np.cos(nu1)
y_leg1 = r_nu1 * np.sin(nu1)

# Leg 2: half ellipse, departure at +x (nu=0), arrival at -x (nu=pi)
nu2 = np.linspace(0, np.pi, 900)
r_nu2 = p_leg2 / (1 + e_leg2 * np.cos(nu2))
x_leg2 = r_nu2 * np.cos(nu2)
y_leg2 = r_nu2 * np.sin(nu2)

# Orbit rings
th = np.linspace(0, 2*np.pi, 1200)

# Frames
N1 = 110; N2 = 130; N = N1 + N2
t1 = np.linspace(0, T_leg1_sec, N1)
t2 = np.linspace(0, tof_leg2_sec, N2)
t_all = np.concatenate([t1, T_leg1_sec + t2])

# Earth starts at +x, drifts forward (resonant → returns close to same spot)
theta_E_all = n_E * t_all

# Jupiter: must be at -x (angle=pi) when spacecraft arrives at end of Leg 2
theta_J_end = np.pi
theta_J0    = theta_J_end - n_J * (T_leg1_sec + tof_leg2_sec)
theta_J_all = theta_J0 + n_J * t_all

x_E_all = r_E * np.cos(theta_E_all)
y_E_all = r_E * np.sin(theta_E_all)
x_J_all = r_J * np.cos(theta_J_all)
y_J_all = r_J * np.sin(theta_J_all)

idx1 = np.linspace(0, len(nu1)-1, N1).astype(int)
idx2 = np.linspace(0, len(nu2)-1, N2).astype(int)
x_sc_all = np.concatenate([x_leg1[idx1], x_leg2[idx2]])
y_sc_all = np.concatenate([y_leg1[idx1], y_leg2[idx2]])

fig, ax = plt.subplots(figsize=(7, 7))
ax.set_aspect("equal")
ax.set_xlabel("x (au)"); ax.set_ylabel("y (au)")
ax.set_title("earth → earth flyby → jupiter  (EEJ, ideal patched conic)")
ax.grid(True, alpha=0.3)

ax.plot(0, 0, "yo", ms=10)
ax.text(0.08, 0.08, "sun", fontsize=8)
ax.plot(r_E*np.cos(th)*to_au, r_E*np.sin(th)*to_au, "b-",  lw=0.8, label="earth orbit")
ax.plot(r_J*np.cos(th)*to_au, r_J*np.sin(th)*to_au, "g-",  lw=0.8, label="jupiter orbit")
ax.plot(x_leg1*to_au, y_leg1*to_au, "c--", lw=0.8, alpha=0.5, label="leg 1 (resonant)")
ax.plot(x_leg2*to_au, y_leg2*to_au, "m--", lw=0.8, alpha=0.5, label="leg 2 → jupiter")

ax.plot([r_E*to_au], [0], "bx", ms=9)
ax.text(r_E*to_au + 0.05, 0.05, "depart /\nflyby", fontsize=7)
ax.plot([-r_J*to_au], [0], "gx", ms=9)
ax.text(-r_J*to_au - 0.05, 0.05, "arrive", fontsize=7, ha="right")

E_dot,  = ax.plot([], [], "bo", ms=7,  label="earth")
J_dot,  = ax.plot([], [], "go", ms=8,  label="jupiter")
sc_dot, = ax.plot([], [], "ro", ms=5,  label="spacecraft")
sc_tr,  = ax.plot([], [], "r-", lw=1.5)
info_tx = ax.text(0.02, 0.02, "", transform=ax.transAxes, fontsize=8,
                  bbox=dict(facecolor="white", alpha=0.8, edgecolor="gray"))

pad = 0.8
lim = r_J*to_au + pad
ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
ax.legend(loc="upper right", fontsize=7)

def update(k):
    E_dot.set_data([x_E_all[k]*to_au], [y_E_all[k]*to_au])
    J_dot.set_data([x_J_all[k]*to_au], [y_J_all[k]*to_au])
    sc_dot.set_data([x_sc_all[k]*to_au], [y_sc_all[k]*to_au])
    sc_tr.set_data(x_sc_all[:k+1]*to_au, y_sc_all[:k+1]*to_au)
    days = t_all[k] / 86400.0
    leg  = "leg 1: resonant loop" if k < N1 else "leg 2: → jupiter"
    info_tx.set_text(f"t = {days:.0f} d\n{leg}")
    return E_dot, J_dot, sc_dot, sc_tr, info_tx

anim = FuncAnimation(fig, update, frames=N, interval=50, blit=True)
anim.save("EEJ_transfer.gif", writer=PillowWriter(fps=20))
plt.close(fig)
print("\nwrote: EEJ_transfer.gif")